In [3]:
# Q1. Data Loading & Preprocessing

import pandas as pd

# Load dataset
df = pd.read_csv('heart.csv')

# Separate features (X) and target (y)
X = df.drop(columns=['HeartDisease'])
y = df['HeartDisease']

# One-hot encoding for categorical variables
X = pd.get_dummies(X, drop_first=True)
print("Data Loading & Preprocessing Complete!")

# Q2. Train-Test Split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")


# Q3. Building Logistic Regression Model

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
print("Model Training Completed!")


# Q4. Making Predictions

y_pred = model.predict(X_test)

print("First 10 Actual Values (y_test):")
print(y_test.iloc[:10].values)

print("\nFirst 10 Predicted Values (y_pred):")
print(y_pred[:10])


# Q5. Confusion Matrix

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(cm)
print(f"\nTrue Negative (TN): {tn}")
print(f"False Positive (FP): {fp}")
print(f"False Negative (FN): {fn}")
print(f"True Positive (TP): {tp}")


# Q6. Model Evaluation Metrics

from sklearn.metrics import classification_report

print("Classification Report:")
print(classification_report(y_test, y_pred))


# Q7. Saving the Model & Objects

import joblib

joblib.dump(model, 'heart_model.pkl')
joblib.dump(X.columns.tolist(), 'columns.pkl')
print("Model and columns saved successfully!")


# Q8. Loading & Testing Saved Model

# Load saved model & columns
loaded_model = joblib.load('heart_model.pkl')
loaded_columns = joblib.load('columns.pkl')

# Sample input testing
sample_patient = X_test.iloc[0:1]
sample_pred = loaded_model.predict(sample_patient)

print(f"Test Prediction on Sample Data: {sample_pred[0]}")

Data Loading & Preprocessing Complete!
X_train shape: (734, 15)
X_test shape: (184, 15)
y_train shape: (734,)
y_test shape: (184,)
Model Training Completed!
First 10 Actual Values (y_test):
[0 1 1 1 0 1 1 0 1 1]

First 10 Predicted Values (y_pred):
[0 0 1 1 0 1 1 0 1 1]
Confusion Matrix:
[[67 10]
 [17 90]]

True Negative (TN): 67
False Positive (FP): 10
False Negative (FN): 17
True Positive (TP): 90
Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.87      0.83        77
           1       0.90      0.84      0.87       107

    accuracy                           0.85       184
   macro avg       0.85      0.86      0.85       184
weighted avg       0.86      0.85      0.85       184

Model and columns saved successfully!
Test Prediction on Sample Data: 0


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [4]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib

# Page configuration
st.set_page_config(page_title="Heart Disease Predictor", layout="centered")

st.title(" Heart Disease Prediction App")
st.write("Enter patient metrics below to predict the likelihood of heart disease.")

# Load saved artifacts
model = joblib.load('heart_model.pkl')
model_columns = joblib.load('columns.pkl')

# Q9: Streamlit Input Interface
st.subheader("Patient Details")

col1, col2 = st.columns(2)

with col1:
    age = st.number_input("Age", min_value=1, max_value=120, value=50)
    resting_bp = st.number_input("Resting BP (mm Hg)", min_value=50, max_value=250, value=120)

with col2:
    cholesterol = st.number_input("Cholesterol (mm/dl)", min_value=0, max_value=600, value=200)
    max_hr = st.number_input("Max HR", min_value=60, max_value=220, value=150)

# Q10: Complete App with Prediction
if st.button("Predict Heart Disease"):
    # Create raw dataframe from input
    raw_data = {
        'Age': [age],
        'RestingBP': [resting_bp],
        'Cholesterol': [cholesterol],
        'MaxHR': [max_hr]
    }

    input_df = pd.DataFrame(raw_data)
    input_df = pd.get_dummies(input_df)

    # Align columns with training data
    input_df = input_df.reindex(columns=model_columns, fill_value=0)

    # Make Prediction
    prediction = model.predict(input_df)[0]

    st.divider()
    if prediction == 1:
        st.error("Heart Disease Detected: Yes")
    else:
        st.success("Heart Disease Detected: No")

Writing app.py


In [ ]:
# Install Streamlit & Cloudflare
!pip install streamlit -q
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# Run Streamlit & Tunnel
!streamlit run app.py & ./cloudflared tunnel --url http://localhost:8501

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 76.8 MB/s eta 0:00:00
--2026-07-28 14:39:35--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.116.3
Connecting to github.com (github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.7.3/cloudflared-linux-amd64 [following]
--2026-07-28 14:39:35--  https://github.com/cloudflare/cloudflared/releases/download/2026.7.3/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/3812f7fa-ce13-4147-a9fb-197be83d49fa?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-07-28T15%3A38%3A05Z&rscd=attachment%3B+filename%3D